# 03 - Erro com pontos flutuantes
Vamos aprender sobre como usar os erros de ponto flutuante para resolução de problemas numéricos.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana3
```

Faça o checkout nessa nova branch:

```bash
git checkout semana3
```

<hr />

## Atividade 1
A função exponencial natural pode ser definida pelo limite:
$$
e^x=\lim_{n\to\infty}\left(1+\frac{x}{n}\right)^n,
$$
mas também é dada pela **série de Maclaurin**:
$$
e^x=\sum_{n=0}^{\infty}\frac{x^n}{n!}
=1+\frac{x}{1!}+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots
$$

Implemente em **Python** o cálculo de $e^x$ pela série, interrompendo a soma quando o termo ficar menor que o limite prático de contribuição, usando a precisão de máquina como critério.

In [7]:
import math
import sys

def exp_series(x, atol=0.0):
    """ Aproxima e^x pela série de Maclaurin com critério de parada numérico. """
    eps = sys.float_info.epsilon
    s = 1.0
    term = 1.0
    n = 0
    tol_abs = max(atol, eps)

    while True:
        n += 1
        term *= x / n
        s += term
        if abs(term) < eps * abs(s) or abs(term) < tol_abs:
            break
        if n > 10_000:
            break
    return s, n, term

def main():
    for val in [1.0, 5.0, -2.0]:
        approx, nterms, last = exp_series(val)
        print(f"x={val:+g} -> e^x ≈ {approx:.16g} (math.exp={math.exp(val):.16g}, termos={nterms})")

if __name__ == "__main__":
    main()

x=+1 -> e^x ≈ 2.718281828459046 (math.exp=2.718281828459045, termos=18)
x=+5 -> e^x ≈ 148.4131591025766 (math.exp=148.4131591025766, termos=33)
x=-2 -> e^x ≈ 0.1353352832366127 (math.exp=0.1353352832366127, termos=24)


## Atividade 2

Implemente:
$$
e^x\approx\left(1+\frac{x}{n}\right)^n
$$
com $n$ crescente, e:

1. Explique por que, para $x<0$ e $n$ muito grande, pode ocorrer **cancelamento catastrófico**;
2. Proponha um critério de parada numérico para encerrar o crescimento de $n$ sem perder precisão.

**Análise:** na avaliação direta, $x/n$ pode ficar menor que a resolução do tipo `float`; então $1+x/n$ é arredondado para 1 e a informação de $x/n$ desaparece. A implementação usa `log1p(x/n)`, que preserva essa parcela pequena, e para quando duas aproximações sucessivas diferem apenas dentro das tolerâncias absoluta/relativa (ou ficam numericamente idênticas).


In [8]:
import math
import sys

def exp_limit(x, n0=10, n_max=10_000_000, growth=2.0, rtol=1e-14, atol=0.0):
    """Aproxima exp(x) pelo limite, usando log1p para evitar cancelamento."""
    n = max(1, n0, math.floor(-x) + 1)
    previous = None

    while n <= n_max:
        value = math.exp(n * math.log1p(x / n))
        if previous is not None:
            error = abs(value - previous)
            if error <= max(atol, rtol * abs(value)):
                return value, n
            if value == previous:
                return value, n
        previous = value
        n = max(n + 1, int(n * growth))

    return previous, n_max

for value in [1.0, 5.0, -2.0]:
    approximation, n = exp_limit(value)
    print(value, approximation, math.exp(value), n)


1.0 2.7182815692235343 2.718281828459045 10000000
5.0 148.4128052586783 148.4131591025766 10000000
-2.0 0.1353352316102959 0.1353352832366127 10000000


## Atividade 3

Para $|x|$ grande, use:
$$
e^x = \left(e^{m\cdot 2^{-k}}\right)^{2^k}, \quad
k = \left\lceil \log_2\!\left(\frac{|x|}{\theta}\right)\right\rceil, \quad m = \frac{x}{2^k}
$$
Calcule $e^{m}$ pela série (Ex. 1) e depois eleve ao quadrado $k$ vezes.

In [9]:
import math

def exp_series_scaling(x, theta=1.0):
    if x == 0.0:
        return 1.0, 0, 0.0, 0
    k = max(0, math.ceil(math.log2(abs(x) / theta))) if abs(x) > theta else 0
    m = x / (2**k)

    em, n_terms, _ = exp_series(m)
    y = em
    for _ in range(k):
        y *= y

    return y, k, n_terms

def main():
    for val in [10.0, -20.0]:
        y, k, n = exp_series_scaling(val, theta=1.0)
        print(
            f"x={val:+g} -> e^x ≈ {y:.6e} (math.exp={math.exp(val):.6e})  [k={k}, termos série(m)={n}]"
        )

if __name__ == "__main__":
    main()

x=+10 -> e^x ≈ 2.202647e+04 (math.exp=2.202647e+04)  [k=4, termos série(m)=16]
x=-20 -> e^x ≈ 2.061154e-09 (math.exp=2.061154e-09)  [k=5, termos série(m)=16]


## Atividade 4

Use:
$$
\cos x=\sum_{n=0}^{\infty}(-1)^n\frac{x^{2n}}{(2n)!}
$$
com a recursão:
$$
t_{n+1}=t_n\cdot\frac{-x^2}{(2n+1)(2n+2)}
$$
Defina um critério de parada baseado em `epsilon` e compare o erro relativo para $x\in[-20,20]$ (200 pontos) contra `math.cos(x)`.

In [10]:
import math
import sys

def cos_series(x, rtol=1e-15, atol=0.0):
    """Calcula cos(x) por Maclaurin após redução estável do argumento."""
    reduced = math.remainder(x, 2.0 * math.pi)
    total = 1.0
    term = 1.0
    n = 0

    while n < 10_000:
        term *= -(reduced * reduced) / ((2*n + 1) * (2*n + 2))
        updated = total + term
        n += 1
        if updated == total or abs(term) <= max(atol, rtol * abs(updated), sys.float_info.epsilon):
            return updated, n
        total = updated

    raise RuntimeError("A série do cosseno não convergiu.")

xs = [-20.0 + 40.0*i/199 for i in range(200)]
relative_errors = []
for x in xs:
    reference = math.cos(x)
    approximation, _ = cos_series(x)
    relative_errors.append(abs(approximation-reference) / max(abs(reference), sys.float_info.epsilon))

print(f"erro relativo máximo = {max(relative_errors):.3e}")


xs=[-20.0, -19.798994974874372, -19.597989949748744, -19.396984924623116, -19.195979899497488, -18.99497487437186, -18.79396984924623, -18.592964824120603, -18.391959798994975, -18.190954773869347]
errs=[5.703136296553168e-10, 3.517699465049873e-10, 1.6916149592205443e-09, 2.054554304464773e-10, 1.436952778988143e-10, 4.906094419609985e-10, 1.5660149843554905e-09, 8.159538600338578e-10, 8.390020900250761e-10, 9.075207252351447e-11]...


## Atividade 5

Dado $x$ e uma tolerância $\tau$, encontre o menor $N$ tal que:
$$
R_{N+1}(x)=\sum_{n=N+1}^{\infty}\frac{|x|^n}{n!} < \tau
$$

In [11]:
from decimal import Decimal, localcontext

def min_terms_for_tol(x, tol=1e-12, precision=80):
    """Menor N tal que sum_(n=N+1)^inf |x|^n/n! < tol."""
    if tol <= 0:
        raise ValueError("A tolerância deve ser positiva.")

    with localcontext() as context:
        context.prec = precision
        value = Decimal(str(abs(x)))
        tolerance = Decimal(str(tol))
        exponential = value.exp()
        partial = Decimal(1)
        term = Decimal(1)
        n = 0

        while exponential - partial >= tolerance:
            n += 1
            term *= value / n
            partial += term
            if n > 100_000:
                raise RuntimeError("Não houve convergência.")

        return n

for value in [1, 3, 10]:
    n = min_terms_for_tol(value, 1e-12)
    print(f"x={value}: N={n}")


x=1: ~15 termos para atingir tol=1e-12
x=3: ~24 termos para atingir tol=1e-12
x=10: ~47 termos para atingir tol=1e-12


## Atividade 6

Usando `decimal` ou `mpmath`, compute $e^x$ em alta precisão e compare com o resultado de `float64` (Ex. 1) para $x\in\{20, 40, 50\}$.
Analise:
- perda de dígitos significativos;
- quando o `float64` começa a saturar por overflow.


In [12]:
import math
import sys
from decimal import Decimal, localcontext

def exp_decimal(x, precision=80):
    with localcontext() as context:
        context.prec = precision
        return +Decimal(str(x)).exp()

def compare_float_vs_highprecision(xs=(20, 40, 50), precision=80):
    rows = []
    for x in xs:
        high_precision = exp_decimal(x, precision)
        try:
            float64 = math.exp(x)
        except OverflowError:
            float64 = float("inf")
        relative_error = (
            abs((Decimal.from_float(float64) - high_precision) / high_precision)
            if math.isfinite(float64) else None
        )
        rows.append((x, high_precision, float64, relative_error))
    return rows

for x, high_precision, float64, relative_error in compare_float_vs_highprecision():
    print(f"x={x}: float64={float64:.16e}, erro relativo={relative_error:.3E}")

overflow_limit = math.log(sys.float_info.max)
print(f"limite de overflow do float64: x ≈ {overflow_limit:.12f}")
for x in [709.0, overflow_limit, 710.0]:
    try:
        print(f"exp({x}) = {math.exp(x)}")
    except OverflowError:
        print(f"exp({x}) = overflow")


x=20  high-prec=485165195.40979027...  float64=485165195.4097903  erro_rel=4.54E-17
x=40  high-prec=235385266837019985...  float64=2.3538526683702e+17  erro_rel=6.20E-17
x=50  high-prec=518470552858707246...  float64=5.184705528587072e+21  erro_rel=8.95E-17


## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 3"
git push origin semana3
```